# Deep Dive EDA: Feature Matrix Insights
This notebook analyzes the relationship between road geometry, urban features, traffic speed, and accident risk.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set visual style
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_palette("magma")

## 1. Data Loading
Loading the joined feature matrix and road segments.

In [ ]:
matrix_path = "../data/processed/features/feature_matrix.parquet"
roads_path = "../data/processed/road_segments.gpkg"

df = pd.read_parquet(matrix_path)
gdf_roads = gpd.read_file(roads_path)
gdf = gdf_roads.merge(df, on="segment_id", how="inner")

print(f"Analyzing {len(df):,} road segments.")

## 2. Top 10 Most Dangerous Roads
Which streets in Bangkok have the highest total accident counts?

In [ ]:
top_roads = gdf.groupby('name')['acc_total'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_roads.values, y=top_roads.index, palette='Reds_r')
plt.title('Top 10 Most Dangerous Streets (2019-2026 Total)')
plt.xlabel('Total Accident Count')
plt.ylabel('Street Name')
plt.show()

## 3. The Junction Effect (Intersections)
Do accidents happen closer to intersections? We compare the distance to intersections for segments with vs. without accidents.

In [ ]:
df['has_accident'] = df['acc_total'] > 0

plt.figure(figsize=(10, 6))
sns.kdeplot(data=df, x='dist_intersection_m', hue='has_accident', fill=True, common_norm=False)
plt.title('Distance to Intersection: Accidents vs. No Accidents')
plt.xlabel('Distance to Nearest Intersection (meters)')
plt.xlim(0, 500) # Most junctions are close
plt.show()

## 4. Speed vs. Accident Severity
How does traffic speed affect the frequency of accidents?

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df[df['acc_total'] > 0], x='speed_mean', y='acc_total', alpha=0.3, color='teal')
plt.title('Average Speed vs. Accident Count')
plt.xlabel('Mean Traffic Speed (km/h)')
plt.ylabel('Total Accidents')
plt.show()

## 5. Urban Context: POIs and Buildings
Comparing building density and POI counts to accident risk.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.regplot(data=df.sample(5000), x='building_density_200m', y='acc_total', ax=ax[0], scatter_kws={'alpha':0.1}, line_kws={'color':'red'})
ax[0].set_title('Building Density vs. Accident Risk')

sns.boxplot(data=df, x='poi_count_200m', y='acc_total', ax=ax[1])
ax[1].set_title('POI Count (200m) vs. Accident Count')
ax[1].set_ylim(0, 10)

plt.show()

## 6. Temporal Risk Fingerprint
When do most accidents happen across the city?

In [ ]:
temporal_cols = ['acc_morning_peak', 'acc_daytime', 'acc_evening_peak', 'acc_late_night']
temporal_sum = df[temporal_cols].sum()

plt.figure(figsize=(10, 6))
temporal_sum.plot(kind='pie', autopct='%1.1f%%', colors=sns.color_palette('viridis'))
plt.title('Accident Distribution by Time of Day')
plt.ylabel('')
plt.show()